In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from scipy.stats import rankdata
import os
import random



os.environ["PYTHONHASHSEED"] = str(42)
random.seed(42)
np.random.seed(42)

def normalize(x):
    x = np.asarray(x, dtype=np.float64)
    x = x / x.sum(axis=1, keepdims=True)
    pseudo = x[x > 0].min() / 2
    x = np.where(x == 0, pseudo, x)
    x = x / x.sum(axis=1, keepdims=True)
    log_x = np.log(x)
    return log_x - log_x.mean(axis=1, keepdims=True)


tr = pd.read_csv("train.csv")
te = pd.read_csv("test.csv")

feats = [c for c in tr.columns if c not in ["sample_id", "label", "cohort", "age", "gender", "bmi", "library_size", "sex_m"]]


y = tr["label"].values

models = {
    "et": ExtraTreesClassifier(n_estimators=500, max_features=0.1, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=42),
    "lr_l2": LogisticRegression(C=0.7, l1_ratio=0.0, class_weight="balanced", max_iter=5000, random_state=42),
    "xgb": XGBClassifier(n_estimators=1000, learning_rate=0.06, max_depth=3, min_child_weight=0.86, colsample_bytree=0.15, subsample=0.65, reg_lambda=0.22, random_state=42,
                         scale_pos_weight=(y == 0).sum() / (y == 1).sum(), tree_method="hist", n_jobs=-1, eval_metric="logloss"),
}

tr_q = normalize(tr[feats].astype(float).values)
te_q = normalize(te[feats].astype(float).values)

preds = {}
for name, model in models.items():
    model.fit(tr_q, y)
    preds[name] = model.predict_proba(te_q)[:, 1]

ens = np.mean([rankdata(p) for p in preds.values()], axis=0) / len(te)

df_new = pd.DataFrame({"sample_id": te["sample_id"], "label": ens})
df_new.to_csv("submission_fixed.csv", index=False)
